# 12.8 Resilience: fault injection and the detection rate

Component testing asks whether each tool is correct; the resilience probe asks what happens when a tool fails. A gateway is installed on the agent's executor and one tool at a time is made to error; the measured quantity is the **detection rate** -- the fraction of runs in which the agent escalates rather than silently proceeding on a corrupted result. This notebook reads the pinned resilience numbers and checks them.

**Reference (run separately):** the real fault-injection probe installs a `ToolGateway` on the capstone agent's executor, errors one tool per run, and records whether the agent escalated (detected) or proceeded (silent).

```python
from agentlab.harness.capstone import CapstoneTestHarness

harness = CapstoneTestHarness(agent=capstone_agent, cases=eval_cases)

# fault_injection() installs a ToolGateway on the executor that forces one
# tool at a time to raise; the agent's run is observed for escalation.
report = harness.fault_injection(fault='error')

print(report.detection_rate)      # fraction of faulted runs that escalate
print(report.silent_failures)     # runs that proceeded on a corrupted result
print(report.per_tool_detection)  # detection rate broken out per tool
```

This call is GPU-heavy and not run here; its outputs are pinned in `capstone_companions.json`.

## What the probe injects, and what counts as detection

The probe targets one tool at a time. For each of the five tools it installs a `ToolGateway` carrying a single `FaultProfile` matched to that tool, runs the scenarios, then uninstalls it and moves on, so the other four tools behave normally throughout. The fault itself is one of three kinds:

- **hard error** (the default, and the one reported here) -- `error_rate=1.0`, so the gateway blocks every call to the tool and it fails whenever it is reached;
- **latency** -- the call is delayed by a fixed budget, then allowed;
- **stale** -- the call is let through but its result is replaced by an out-of-date response, the case a syntactic check cannot catch.

```python
from knowlytix.harness.testing import FaultProfile, ToolGateway
# the default fault: force one tool to fail on every call it receives
profile = FaultProfile(tool_pattern='search_policy', error_rate=1.0,
                       error_message='search_policy service unavailable (injected)')
```

A faulted run counts as **detected** only when the agent both *reached* the faulted tool and then *failed loud*, ending escalated or failed rather than finishing on a corrupted result. A run that finishes normally after hitting the broken tool is a **silent** failure, the outcome a governed agent must never produce. The detection rate is measured over the runs that reached the faulted tool, so a run that escalated earlier never counts against that tool.

**What the next cell does** — loads the pinned resilience block from disk:

1. **Find the data root.** Walk up from `Path.cwd()` until a directory containing `data/capstone_companions.json` is found.
2. **Read the artifact.** `json.loads(...)` parses that file into `C`, then `res = C['resilience']` pulls the resilience sub-dict (`fault`, `detection_rate`, `silent_failures`, `per_tool_detection`).
3. **Echo it.** The bare `res` displays the dict so the raw numbers are visible.

In [ ]:
import json
from pathlib import Path

root = Path.cwd().parent / 'code'
while not (root / 'data' / 'capstone_companions.json').exists() and root != root.parent:
    root = root.parent

C = json.loads((root / 'data' / 'capstone_companions.json').read_text())
res = C['resilience']
res

**What the next cell does** — prints the resilience numbers in a readable layout:

1. **Headline figures.** Prints `res['fault']`, `res['detection_rate']` and `res['silent_failures']`.
2. **Per-tool breakdown.** Loops over `res['per_tool_detection'].items()` and prints the detection rate for each of the five tools.

In [ ]:
print(f"fault injected        : {res['fault']}")
print(f"detection rate (all)  : {res['detection_rate']:.2f}")
print(f"silent failures       : {res['silent_failures']}")
print()
print('per-tool detection rate:')
for tool, rate in res['per_tool_detection'].items():
    print(f"  {tool:18s} {rate:.2f}")

On this agent the detection rate is **1.0 on every tool** and the silent-failure count is **zero**: a hard fault on any step makes the agent fail loud and escalate to a human. The gateway establishes that an outright failure is caught; the verifiers of the earlier chapters -- the exact-register check and the groundedness distance -- are what catch the harder case of a plausible-but-wrong structured result.

## The hardest fault: a stance reversal

The purest plausible-but-wrong result is a draft that **inverts** a policy's stance -- naming the right entity and relation but asserting *optional* where the policy makes it required, or *permitted* where it is forbidden. The type and shape are correct, so a syntactic check passes; only a comparison against the stored stance catches it. The value-polarity verifier makes that comparison and returns `supported`, `contradicted` or `uncertain`.

**What the next cell does** — reads the pinned value-polarity result (`C['value_polarity']`) and prints the reversal-detection and synonym-acceptance rates.

In [ ]:
vp = C['value_polarity']
print(f"stance reversals flagged contradicted : {vp['reversal_detected']}/{vp['reversal_n']}"
      f"  (rate {vp['reversal_rate']:.2f})")
print(f"synonym stances accepted as supported : {vp['synonym_accepted']}/{vp['synonym_n']}"
      f"  (rate {vp['synonym_rate']:.2f})")

On a four-item reversal battery the verifier flags three of the four reversals as contradicted; the miss is a required-to-optional assertion scored `uncertain`, a deferral to review rather than a false pass. On the matching synonym battery, where the asserted stance paraphrases the stored one, it accepts three of four. The stance reversal is the plausible-but-wrong fault in its purest form, and it is caught downstream, not at the tool boundary.

**What the next cell does** — asserts the resilience contract holds:

1. **Headline checks.** Asserts `res['fault'] == 'error'`, `res['detection_rate'] == 1.0` and `res['silent_failures'] == 0`.
2. **Coverage check.** Asserts the keys of `res['per_tool_detection']` are exactly the five capstone tools.
3. **Per-tool check.** Asserts every value in `res['per_tool_detection']` is `1.0`, then prints `self-check passed`.

In [ ]:
assert res['fault'] == 'error'
assert res['detection_rate'] == 1.0, 'agent must escalate on every faulted run'
assert res['silent_failures'] == 0, 'no fault may propagate silently'
assert set(res['per_tool_detection']) == {
    'classify_complaint', 'extract_facts', 'search_policy',
    'flag_regulatory', 'draft_response',
}
assert all(r == 1.0 for r in res['per_tool_detection'].values()), \
    'every tool fault must be detected'
vp = C['value_polarity']
assert vp['reversal_detected'] == 3 and vp['reversal_n'] == 4   # 3/4 reversals caught
assert vp['synonym_accepted'] == 3 and vp['synonym_n'] == 4     # 3/4 synonyms accepted
print('self-check passed')